In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

In [ ]:
load_dotenv()
model = ChatOpenAI()

In [ ]:
# create a state
class BlogState(TypedDict):
  title: str
  outline: str
  content: str

In [ ]:
def create_outline(state: BlogState) -> BlogState:
  # fetch title
  title = state['title']

  # call llm gen outline
  prompt = f'Generate a detailed outline for a blog on the topic - {title}'
  outline = model.invoke(prompt).content

  # update state
  state['outline'] = outline

  return state

def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the following outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [ ]:
# create our graph
graph = StateGraph(BlogState)

# add nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# add edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

# compile
workflow = graph.compile()

In [ ]:
final_state = workflow.invoke({ 'title': 'Moons of Jupiter' })
print(final_state['outline'], final_state['content'])